# ViBioMIR · phần 3/3 · chạy trên tài khoản Kaggle của bạn

Notebook tự chứa toàn bộ mã, xử lý **1.626.231 chunk** bằng **hai GPU T4**.
Dataset nguồn đã gắn sẵn trên Kaggle; không cần tải/upload lại dữ liệu,
GitHub, token Hugging Face hoặc output phần 1/2.

**Người sở hữu:** cấp quyền truy cập cho bạn ở cả notebook này và dataset private
`nhminh107/vibiomir-chunks-qwen3-input-3parts-20261024` (hai quyền độc lập).

**Người nhận:**
1. Mở notebook được chia sẻ. Để chạy bản riêng trên tài khoản của bạn, chọn **Copy & Edit**.
2. Kiểm tra dataset nguồn vẫn có trong bảng Input. Nếu cần, **Add Input** dataset
   đã được cấp quyền; chỉ gắn một bản của nguồn này.
3. Chọn **GPU T4 ×2**, bật **Internet**, giữ nguyên cấu hình.
4. **Save Version → Save & Run All**. Không phải sửa đường dẫn hay username.

Cell kiểm tra đầu vào sẽ dừng sớm nếu thiếu file, gắn trùng nguồn hoặc không đủ hai T4.
Mỗi GPU giữ một bản mô hình Qwen3 Embedding 0.6B, xử lý một dải dữ liệu riêng.
Giữ nguyên `CONFIG['part'] = 3` và cấu hình encoder để output khớp phần 1/2.

Kết quả: `/kaggle/working/vibiomir_embeddings/part-3/`, Parquet float32 1024 chiều,
giữ toàn bộ ID/metadata gốc. Dừng có checkpoint sau 11,25 giờ nếu chưa xong.
Chỉ báo hoàn tất khi `manifest.json` có `complete: true`, `rows: 1626231`.

**Gửi kết quả lại:** tải toàn bộ output sau khi version chạy xong, bao gồm Parquet,
receipt `*.parquet.json`, `manifest.json` và báo cáo hai GPU. Không chỉ gửi notebook.
Hướng dẫn chi tiết và cách tiếp tục checkpoint có trong `README.md` đi kèm.


In [ ]:
import time
import json
from pathlib import Path
CONFIG = json.loads('{"part": 3, "model_id": "Qwen/Qwen3-Embedding-0.6B", "revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3", "dimension": 1024, "max_length": 4096, "dtype": "float16", "pooling": "last_token", "normalize": true, "document_prompt": "", "vector_name": "qwen3_dense", "output_rows": 8192, "max_batch_size": 64, "padded_token_budget": 16384, "max_hours": 11.25, "expected_source_sha256": "a45123b3de7a9b048f83418515a3ec6b3dd09e1e4491dd361429b787659a3e7f", "input_root": "/kaggle/input", "output_root": "/kaggle/working/vibiomir_embeddings", "resume_roots": [], "require_t4": true}')
CONFIG['deadline_unix'] = time.time() + CONFIG['max_hours'] * 3600
# RESUME: điền thư mục part-N của output cũ đã Add Input.
# CONFIG['resume_roots'] = ['/kaggle/input/.../vibiomir_embeddings/part-3']


In [ ]:
# Không cần sửa username hoặc tên dataset khi chuyển sang tài khoản khác.
import torch

if CONFIG['part'] != 3:
    raise ValueError('Notebook bàn giao này chỉ chạy phần 3; giữ CONFIG[part] = 3.')
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Hãy chọn Accelerator = GPU T4 ×2 trước khi chạy.')
devices = [torch.cuda.get_device_name(index) for index in (0, 1)]
if any('T4' not in name for name in devices):
    raise RuntimeError(f'Cần hai T4; GPU hiện tại: {devices}')

matches = []
for candidate in Path(CONFIG['input_root']).rglob('source_manifest.json'):
    data = json.loads(candidate.read_text())
    if data.get('source_sha256') == CONFIG['expected_source_sha256']:
        matches.append((candidate, data))
if len(matches) != 1:
    raise ValueError('Cần được cấp quyền dataset nguồn và Add Input đúng một bản chứa '
                     'source_manifest.json, chunks-04.jsonl.zst và chunks-05.jsonl.zst.')
source_manifest_path, source_info = matches[0]
part_shards = [item for item in source_info['shards'] if item['part'] == 3]
if (sorted(item['gpu'] for item in part_shards) != [0, 1]
        or sum(item['rows'] for item in part_shards) != 1626231):
    raise ValueError('Manifest không khớp phần 3 đã chia sẵn.')
missing = [item['file'] for item in part_shards
           if not (source_manifest_path.parent / item['file']).is_file()]
if missing:
    raise FileNotFoundError(f'Dataset thiếu file: {missing}. Upload đủ ba file trong input/.')
print(json.dumps({'part': 3, 'expected_rows': 1626231, 'gpus': devices,
                  'input': str(source_manifest_path.parent)}, ensure_ascii=False, indent=2))


In [ ]:
# Giữ PyTorch/CUDA có sẵn của Kaggle, chỉ pin các thư viện cần thiết.
import os
os.environ['HF_HOME'] = '/kaggle/temp/vibiomir-hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.1', 'huggingface-hub==0.36.0',
                'tokenizers==0.22.1', 'pyarrow==20.0.0', 'zstandard==0.23.0',
                'safetensors==0.6.2'], check=True)


In [ ]:
runtime_path = Path('/kaggle/working/vibiomir_embedding_runtime.py')
runtime_path.write_text('"""Two independent T4 workers, resumable Parquet shards, lossless payloads."""\n\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\nimport uuid\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nimport zstandard as zstd\n\n\nPOINT_NAMESPACE = uuid.uuid5(uuid.NAMESPACE_URL, "https://vibiomir.org/chunks")\nPROFILE_FIELDS = ("model_id", "revision", "dimension", "max_length", "dtype", "pooling",\n                  "normalize", "document_prompt", "vector_name")\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".partial")\n    with temporary.open("w") as handle:\n        json.dump(value, handle, ensure_ascii=False, indent=2)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    temporary.replace(path)\n\n\ndef encoder_profile(config):\n    profile = {key: config[key] for key in PROFILE_FIELDS}\n    profile.update(attention="sdpa", padding_side="left", truncation=False,\n                   output_dtype="float32", point_id="uuid5(vibiomir namespace, chunk_id)")\n    return profile\n\n\ndef profile_id(profile):\n    return hashlib.sha256(json.dumps(profile, sort_keys=True).encode()).hexdigest()\n\n\ndef point_id(chunk_id):\n    return str(uuid.uuid5(POINT_NAMESPACE, chunk_id))\n\n\ndef source_records(path, shard):\n    line_number = shard["start_line"]\n    offset = shard["start_byte"]\n    raw_hash = hashlib.sha256()\n    with Path(path).open("rb") as compressed:\n        with zstd.ZstdDecompressor().stream_reader(compressed) as reader:\n            for line in io.BufferedReader(reader):\n                raw_hash.update(line)\n                yield json.loads(line), line_number, offset\n                line_number += 1\n                offset += len(line)\n    if (line_number - shard["start_line"], offset, raw_hash.hexdigest()) != (\n            shard["rows"], shard["end_byte"], shard["raw_sha256"]):\n        raise ValueError("Decompressed input does not match source manifest")\n\n\ndef batches_by_length(lengths, max_batch_size, padded_token_budget):\n    batch = []\n    longest = 0\n    for index in sorted(range(len(lengths)), key=lengths.__getitem__):\n        next_longest = max(longest, lengths[index])\n        if batch and (len(batch) >= max_batch_size or\n                      next_longest * (len(batch) + 1) > padded_token_budget):\n            yield batch\n            batch, longest = [], 0\n        batch.append(index)\n        longest = max(longest, lengths[index])\n    if batch:\n        yield batch\n\n\ndef embed_records(records, tokenizer, model, config, device):\n    import torch\n    import torch.nn.functional as functional\n\n    features = tokenizer([record[0]["content"] for record in records], padding=False,\n                         truncation=False, add_special_tokens=True)\n    lengths = [len(ids) for ids in features["input_ids"]]\n    if max(lengths) > config["max_length"]:\n        index = lengths.index(max(lengths))\n        raise ValueError(f"Chunk {records[index][0][\'chunk_id\']} has {lengths[index]} tokens; "\n                         "increase max_length explicitly; no content was truncated")\n    vectors = np.empty((len(records), config["dimension"]), dtype=np.float32)\n    queue = list(batches_by_length(lengths, config["max_batch_size"], config["padded_token_budget"]))\n    oom_retries = 0\n    with torch.inference_mode():\n        while queue:\n            indices = queue.pop(0)\n            inputs = hidden = pooled = normalized = None\n            try:\n                inputs = tokenizer.pad([{key: value[index] for key, value in features.items()}\n                                        for index in indices], padding=True, return_tensors="pt")\n                inputs = {key: value.to(device) for key, value in inputs.items()}\n                hidden = model(**inputs, use_cache=False, return_dict=True).last_hidden_state\n                # Left padding means the final position is a real token for every row.\n                pooled = hidden[:, -1].float()\n                normalized = functional.normalize(pooled, p=2, dim=1)\n                vectors[indices] = normalized.cpu().numpy()\n            except torch.OutOfMemoryError:\n                if len(indices) == 1:\n                    raise\n                oom_retries += 1\n                middle = len(indices) // 2\n                queue[0:0] = [indices[:middle], indices[middle:]]\n            finally:\n                del inputs, hidden, pooled, normalized\n            if oom_retries:\n                torch.cuda.empty_cache()\n    norms = np.linalg.norm(vectors, axis=1)\n    if not np.isfinite(vectors).all() or not np.allclose(norms, 1, atol=2e-5):\n        raise ValueError("Invalid embedding vectors")\n    return vectors, lengths, oom_retries\n\n\ndef parquet_schema(dimension, metadata):\n    return pa.schema([\n        ("point_id", pa.string()), ("chunk_id", pa.string()), ("doc_id", pa.int64()),\n        ("source_line", pa.int64()), ("source_byte", pa.int64()),\n        ("content", pa.large_string()), ("content_origin", pa.string()),\n        ("token_count", pa.int32()), ("input_token_count", pa.int32()),\n        ("content_sha256", pa.string()), ("payload_json", pa.large_string()),\n        ("vector", pa.list_(pa.float32(), dimension)),\n    ], metadata={b"vibiomir_embedding": json.dumps(metadata, sort_keys=True).encode()})\n\n\ndef write_shard(path, records, vectors, lengths, config, source_manifest, gpu):\n    profile = encoder_profile(config)\n    metadata = dict(schema_version="vibiomir-embedding-v1", encoder=profile,\n                    profile_id=profile_id(profile), source_sha256=source_manifest["source_sha256"],\n                    part=config["part"], gpu=gpu)\n    columns = {name: [] for name in parquet_schema(config["dimension"], metadata).names if name != "vector"}\n    for (record, line_number, offset), length in zip(records, lengths):\n        if "_embedding" in record:\n            raise ValueError("Input already contains reserved _embedding metadata")\n        content_hash = hashlib.sha256(record["content"].encode()).hexdigest()\n        payload = dict(record)\n        payload["_embedding"] = dict(source_file=source_manifest["source_file"],\n            source_sha256=source_manifest["source_sha256"], source_line=line_number,\n            source_byte=offset, content_sha256=content_hash, profile_id=metadata["profile_id"],\n            model_id=config["model_id"], revision=config["revision"], part=config["part"],\n            gpu=gpu, input_token_count=length, truncated=False)\n        values = dict(point_id=point_id(record["chunk_id"]), chunk_id=record["chunk_id"],\n                      doc_id=record["doc_id"], source_line=line_number, source_byte=offset,\n                      content=record["content"], content_origin=record.get("content_origin"),\n                      token_count=record.get("token_count"), input_token_count=length,\n                      content_sha256=content_hash, payload_json=json.dumps(payload, ensure_ascii=False))\n        for key, value in values.items():\n            columns[key].append(value)\n    schema = parquet_schema(config["dimension"], metadata)\n    arrays = [pa.array(columns[field.name], type=field.type) for field in schema if field.name != "vector"]\n    arrays.append(pa.FixedSizeListArray.from_arrays(pa.array(vectors.ravel()), config["dimension"]))\n    table = pa.Table.from_arrays(arrays, schema=schema)\n    temporary = Path(str(path) + ".partial")\n    pq.write_table(table, temporary, compression="zstd", compression_level=3,\n                   row_group_size=1024, use_dictionary=["content_origin"])\n    temporary.replace(path)\n    receipt = dict(file=path.name, rows=len(records), sha256=file_hash(path),\n                   start_line=records[0][1], end_line=records[-1][1],\n                   start_byte=records[0][2], part=config["part"], gpu=gpu,\n                   source_sha256=source_manifest["source_sha256"], profile_id=metadata["profile_id"],\n                   min_tokens=min(lengths), max_tokens=max(lengths))\n    atomic_json(Path(str(path) + ".json"), receipt)\n    return receipt\n\n\ndef validate_shard(path, receipt, config, source_manifest):\n    if receipt["profile_id"] != profile_id(encoder_profile(config)):\n        raise ValueError("Checkpoint encoder configuration differs")\n    if receipt["source_sha256"] != source_manifest["source_sha256"]:\n        raise ValueError("Checkpoint corpus differs")\n    if file_hash(path) != receipt["sha256"]:\n        raise ValueError(f"Checkpoint checksum mismatch: {path}")\n    parquet = pq.ParquetFile(path)\n    if parquet.metadata.num_rows != receipt["rows"]:\n        raise ValueError("Checkpoint row count differs")\n    if receipt["rows"] != receipt["end_line"] - receipt["start_line"] + 1:\n        raise ValueError("Checkpoint has inconsistent source range")\n    metadata = json.loads(parquet.schema_arrow.metadata[b"vibiomir_embedding"])\n    if metadata["profile_id"] != receipt["profile_id"] or metadata["source_sha256"] != receipt["source_sha256"]:\n        raise ValueError("Parquet encoder/source metadata differs")\n    next_line = receipt["start_line"]\n    for batch in parquet.iter_batches(batch_size=1024, columns=["source_line", "vector"]):\n        lines = batch.column(0).to_numpy()\n        if not np.array_equal(lines, np.arange(next_line, next_line + len(lines))):\n            raise ValueError("Checkpoint has missing or reordered rows")\n        next_line += len(lines)\n        values = batch.column(1).values.to_numpy().reshape(len(lines), config["dimension"])\n        if not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values, axis=1), 1, atol=2e-5):\n            raise ValueError("Checkpoint contains invalid vectors")\n\n\ndef restore_shards(output, shard, config, source_manifest, gpu):\n    candidates = {}\n    for root in [output, *map(Path, config["resume_roots"])]:\n        if not root.is_dir():\n            raise ValueError(f"Resume directory does not exist: {root}")\n        for path in root.glob(f"p{config[\'part\']}-gpu{gpu}-*.parquet.json"):\n            receipt = json.loads(path.read_text())\n            if Path(receipt["file"]).name != receipt["file"]:\n                raise ValueError("Invalid checkpoint filename")\n            key = receipt["start_line"]\n            if key in candidates and candidates[key][1] != receipt:\n                raise ValueError("Conflicting checkpoints for the same source range")\n            candidates[key] = (path.parent / receipt["file"], receipt)\n    receipts = []\n    next_line = shard["start_line"]\n    for source, receipt in sorted(candidates.values(), key=lambda item: item[1]["start_line"]):\n        if receipt["start_line"] != next_line or receipt["end_line"] > shard["end_line"]:\n            raise ValueError("Checkpoint is not a contiguous prefix of this GPU\'s source range")\n        if receipt["part"] != config["part"] or receipt["gpu"] != gpu:\n            raise ValueError("Checkpoint belongs to another worker")\n        validate_shard(source, receipt, config, source_manifest)\n        destination = output / receipt["file"]\n        if source.resolve() != destination.resolve():\n            temporary = Path(str(destination) + ".partial")\n            shutil.copyfile(source, temporary)\n            temporary.replace(destination)\n            atomic_json(Path(str(destination) + ".json"), receipt)\n        receipts.append(receipt)\n        next_line = receipt["end_line"] + 1\n    return receipts\n\n\ndef run(config):\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoModel, AutoTokenizer\n\n    started = time.time()\n    rank = int(os.environ["LOCAL_RANK"])\n    if int(os.environ.get("WORLD_SIZE", "0")) != 2 or torch.cuda.device_count() != 2:\n        raise RuntimeError("Exactly two visible GPUs and two torchrun workers are required")\n    device = f"cuda:{rank}"\n    torch.cuda.set_device(rank)\n    name = torch.cuda.get_device_name(rank)\n    if config["require_t4"] and "T4" not in name:\n        raise RuntimeError(f"GPU {rank} is {name}; select Kaggle GPU T4 x2")\n    torch.set_num_threads(2)\n    print(f"[GPU {rank}] {name}; loading a dedicated model replica", flush=True)\n    inputs = Path(config["input_root"])\n    manifests = list(inputs.rglob("source_manifest.json"))\n    matches = [(path, json.loads(path.read_text())) for path in manifests]\n    matches = [(path, data) for path, data in matches\n               if data.get("source_sha256") == config["expected_source_sha256"]]\n    if len(matches) != 1:\n        raise ValueError(f"Expected exactly one matching source manifest, found {len(matches)}")\n    manifest_path, source_manifest = matches[0]\n    shard = next(s for s in source_manifest["shards"] if s["part"] == config["part"] and s["gpu"] == rank)\n    source = manifest_path.parent / shard["file"]\n    if file_hash(source) != shard["compressed_sha256"]:\n        raise ValueError("Compressed input checksum mismatch")\n    output = Path(config["output_root"]) / f"part-{config[\'part\']}"\n    output.mkdir(parents=True, exist_ok=True)\n    receipts = restore_shards(output, shard, config, source_manifest, rank)\n    resumed = sum(item["rows"] for item in receipts)\n    # Both workers share HF cache; Hub file locks protect the download.\n    model_path = snapshot_download(config["model_id"], revision=config["revision"],\n                                   allow_patterns=["*.json", "*.txt", "*.safetensors"], max_workers=2)\n    tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")\n    model = AutoModel.from_pretrained(model_path, torch_dtype=torch.float16,\n                                     attn_implementation="sdpa").to(device).eval()\n    if model.config.hidden_size != config["dimension"]:\n        raise ValueError("Unexpected model embedding dimension")\n    pending = []\n    origins = Counter()\n    new_rows = 0\n    total_oom = 0\n    inference_start = time.time()\n\n    def commit(records):\n        nonlocal new_rows, total_oom\n        vectors, lengths, oom = embed_records(records, tokenizer, model, config, device)\n        path = output / f"p{config[\'part\']}-gpu{rank}-{records[0][1]:09d}.parquet"\n        receipt = write_shard(path, records, vectors, lengths, config, source_manifest, rank)\n        receipts.append(receipt)\n        new_rows += len(records)\n        total_oom += oom\n        origins.update(row[0].get("content_origin", "unknown") for row in records)\n        seconds = time.time() - inference_start\n        rate = new_rows / max(seconds, 1)\n        progress = dict(gpu=rank, device=name, rows=resumed + new_rows, expected_rows=shard["rows"],\n                        rows_per_second=round(rate, 2),\n                        eta_hours=round((shard["rows"] - resumed - new_rows) / max(rate, .001) / 3600, 2),\n                        max_allocated_gib=round(torch.cuda.max_memory_allocated(rank) / 2**30, 3))\n        atomic_json(output / f"progress-gpu{rank}.json", progress)\n        print(json.dumps(progress), flush=True)\n\n    for record in source_records(source, shard):\n        if record[1] < shard["start_line"] + resumed:\n            continue\n        if time.time() >= config["deadline_unix"]:\n            break\n        pending.append(record)\n        if len(pending) == config["output_rows"]:\n            commit(pending)\n            pending = []\n    if pending:\n        commit(pending)\n    count = sum(item["rows"] for item in receipts)\n    worker = dict(run_id=config["run_id"], gpu=rank, device=name, rows=count, expected_rows=shard["rows"],\n                  resumed_rows=resumed, new_rows=new_rows, complete=count == shard["rows"],\n                  elapsed_seconds=round(time.time() - started, 2), oom_batch_splits=total_oom,\n                  new_content_origins=dict(origins), files=receipts,\n                  peak_allocated_gib=round(torch.cuda.max_memory_allocated(rank) / 2**30, 3))\n    atomic_json(output / f"worker-gpu{rank}.json", worker)\n    # Release GPU memory before waiting for the other worker\'s final receipt.\n    del model\n    torch.cuda.empty_cache()\n    if rank == 0:\n        other = output / "worker-gpu1.json"\n        while True:\n            other_worker = json.loads(other.read_text()) if other.exists() else {}\n            if other_worker.get("run_id") == config["run_id"]:\n                break\n            time.sleep(2)\n        workers = [worker, other_worker]\n        if any(w["rows"] == 0 for w in workers):\n            raise RuntimeError("Both GPUs must process nonempty source partitions")\n        summary = dict(schema_version="vibiomir-embedding-v1", part=config["part"],\n                       complete=all(w["complete"] for w in workers),\n                       rows=sum(w["rows"] for w in workers),\n                       expected_rows=sum(w["expected_rows"] for w in workers),\n                       source=source_manifest, encoder=encoder_profile(config),\n                       profile_id=profile_id(encoder_profile(config)), gpu_workers=workers,\n                       files=[entry for w in workers for entry in w["files"]],\n                       elapsed_seconds=round(time.time() - started, 2))\n        atomic_json(output / "manifest.json", summary)\n        print(json.dumps(dict(part=config["part"], complete=summary["complete"],\n                              rows=summary["rows"], expected_rows=summary["expected_rows"])), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    arguments = parser.parse_args()\n    run(json.loads(Path(arguments.config).read_text()))\n')
output = Path(CONFIG['output_root']) / f"part-{CONFIG['part']}"
output.mkdir(parents=True, exist_ok=True)
(output / 'README.md').write_text('# Chạy phần 3 ViBioMIR từ notebook được chia sẻ trên Kaggle\n\nNotebook: https://www.kaggle.com/code/nhminh107/vibiomir-qwen3-06b-part3-t4x2-20261024\n\nDataset input đã gắn sẵn:\nhttps://www.kaggle.com/datasets/nhminh107/vibiomir-chunks-qwen3-input-3parts-20261024\n\n## Người sở hữu cấp quyền\n\nDùng Share/Sharing để cấp quyền cho bạn của bạn ở **cả notebook và dataset private**.\nKaggle quản lý hai quyền riêng biệt; chỉ chia sẻ notebook chưa đủ quyền đọc input.\nKhông cần tải dữ liệu xuống hoặc tạo dataset mới.\n\n## Người nhận chạy\n\n1. Mở notebook, chọn **Copy & Edit** để có bản chạy riêng trên tài khoản của mình.\n2. Kiểm tra Input có dataset nguồn đã được cấp quyền; nếu thiếu, Add Input dataset\n   ở đường dẫn trên. Chỉ gắn một bản nguồn để tránh trùng manifest.\n3. Chọn **GPU T4 ×2**, bật **Internet**. Giữ `CONFIG[\'part\'] = 3` và các cấu hình khác.\n4. **Save Version → Save & Run All**. Notebook tự tìm file, không phải sửa username\n   hoặc đường dẫn. Không cần GitHub, API key hay output phần 1/2.\n\nCell preflight kiểm tra đủ input và hai T4 trước khi tải model. Mỗi GPU giữ một\nmodel Qwen3 Embedding 0.6B riêng. Log phải có cả GPU 0 và GPU 1 cùng tiến độ xử lý.\n\nEncoder giống phần 1/2: FP16 khi chạy, vector lưu FP32, 1024 chiều, last-token\npooling, chuẩn L2. Giữ nguyên toàn bộ ID, nội dung và metadata nguồn.\n\n## Gửi output lại\n\nKết quả nằm ở `/kaggle/working/vibiomir_embeddings/part-3/`.\nChỉ coi là hoàn tất khi `manifest.json` có:\n\n```json\n{"part": 3, "complete": true, "rows": 1626231, "expected_rows": 1626231}\n```\n\nHai worker phải có 825.755 dòng trên GPU 0 và 800.476 dòng trên GPU 1.\nTải toàn bộ output rồi gửi lại người nhờ chạy, hoặc chia sẻ dataset/output private\ncho họ. Giữ đủ Parquet, receipt `*.parquet.json`, manifest và báo cáo hai GPU.\nKhông chỉ gửi notebook vì notebook không chứa các vector output.\n\nNếu `complete: false`, giữ checkpoint. Add Input output cũ, giữ dataset nguồn,\nvà đặt `CONFIG[\'resume_roots\']` thành thư mục `part-3` của output cũ rồi chạy lại.\nNotebook dừng nhận block mới sau 11,25 giờ để chừa thời gian lưu output.\n\nQuyền chia sẻ theo tài liệu Kaggle: https://www.kaggle.com/docs/notebooks\n')
config_path = Path('/kaggle/working/vibiomir_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2))


In [ ]:
# torchrun tạo đúng 2 worker; runtime kiểm tra tên GPU và số chunk từng GPU.
import uuid
CONFIG['run_id'] = str(uuid.uuid4())
config_path.write_text(json.dumps(CONFIG, indent=2))
subprocess.run([sys.executable, '-m', 'torch.distributed.run',
                '--standalone', '--nproc_per_node=2', str(runtime_path),
                '--config', str(config_path)], check=True)


In [ ]:
report = json.loads((output / 'manifest.json').read_text())
print(json.dumps({k: report[k] for k in ('part', 'complete', 'rows', 'expected_rows',
                                       'elapsed_seconds')}, indent=2))
print(json.dumps([{k: worker[k] for k in ('gpu', 'device', 'rows', 'new_rows',
                                          'resumed_rows', 'peak_allocated_gib')}
                  for worker in report['gpu_workers']], indent=2))
if report['complete']:
    print('HOÀN TẤT phần', CONFIG['part'], '— tải toàn bộ output và gửi lại cho người nhờ chạy.')
else:
    print('CHƯA HOÀN TẤT. Đã lưu checkpoint. Add Input output này và đặt resume_roots để tiếp tục.')
